In [13]:
#Load the Vector Store from Disk
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Recreate the embedding function
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2",
                                  model_kwargs={"device": "mps"})

# Load existing Chroma DB
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

#Create a Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

#Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
)

#Create a RAG Prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful customer-support chatbot.
Answer the user's question using ONLY the information provided in the context.
If the answer is not in the context, say "I don't know based on the website content."

Context:
---------
{context}

Question:
---------
{question}

Answer:
---------
""")

import json
from datetime import datetime
from pathlib import Path

LOG_PATH = Path("rag_logs.jsonl")

def log_interaction(question, context, answer):
    record = {
        "timestamp": datetime.utcnow().isoformat(),
        "question": question,
        "context": context,
        "answer": answer,
    }

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


#Create the RAG Chain
def format_docs(docs):
    docs_page_content = []
    for doc in docs:
        page_content = " ".join(doc.page_content.split() )
        docs_page_content.append(page_content)
    return "\n\n".join(doc for doc in docs_page_content)

#Build the chain (LangChain LCEL)
from langchain_core.runnables import RunnablePassthrough

# rag_chain = (
#     {
#         "context": retriever | format_docs,
#         "question": RunnablePassthrough(),
#     }
#     | prompt
#     | llm
# )
def ask(query: str):
    docs = retriever.invoke(query)
    context = format_docs(docs)

    response = llm.invoke(
        prompt.format(
            context=context,
            question=query
        )
    )

    log_interaction(
        question=query,
        context=context,
        answer=response.content
    )

    return response.content


#Ask Questions
response = ask("What does this company offer?")

print(response)


This company offers natural, science-based alternatives to synthetic supplements and promotes accessible, functional foods. They produce highly bioactive, natural folic acid through precision fermentation, which can be used by food, supplement, and pharma brands. They also offer a "Folic Acid supplement" that is natural, active, and organic, MTHFR‑Friendly, with 1,000 mcg / 1 mg and is intended for "Pre‑conception & Prenatal Support." It is described as helping to prevent folate deficiency and neural tube defects.


In [9]:
#Read HF token
with open('../Gemini_token.txt' , 'r') as f:
    lines = f.readlines()
Gemini_token = lines[0]

from openai import OpenAI

client = OpenAI(
    api_key=Gemini_token,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-2.0-flash-lite",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What are the benefits of folic acid?"
        }
    ]
)

print(response.choices[0].message)

ChatCompletionMessage(content="Folic acid is the synthetic form of **folate**, which is a B vitamin (B9). It plays a crucial role in many bodily functions, especially those involving cell growth and DNA synthesis.\n\nHere are the primary benefits of folic acid:\n\n1.  **Prevents Neural Tube Defects (NTDs):** This is arguably its most famous and critical benefit.\n    *   **For Pregnant Women (or those planning pregnancy):** Adequate folic acid intake *before* and during early pregnancy (the first trimester) is essential for the proper development of the baby's brain and spinal cord. It significantly reduces the risk of serious birth defects like spina bifida and anencephaly. Most health organizations recommend 400 mcg daily for women of childbearing age.\n\n2.  **Aids in Red Blood Cell Formation & Prevents Anemia:**\n    *   Folic acid works with vitamin B12 to produce healthy red blood cells.\n    *   A deficiency can lead to **megaloblastic anemia**, where red blood cells are abnorma

In [17]:
print(response.choices[0].message.role)

assistant


In [18]:
print(response.choices[0].message.content)

Folic acid is the synthetic form of **folate**, which is a B vitamin (B9). It plays a crucial role in many bodily functions, especially those involving cell growth and DNA synthesis.

Here are the primary benefits of folic acid:

1.  **Prevents Neural Tube Defects (NTDs):** This is arguably its most famous and critical benefit.
    *   **For Pregnant Women (or those planning pregnancy):** Adequate folic acid intake *before* and during early pregnancy (the first trimester) is essential for the proper development of the baby's brain and spinal cord. It significantly reduces the risk of serious birth defects like spina bifida and anencephaly. Most health organizations recommend 400 mcg daily for women of childbearing age.

2.  **Aids in Red Blood Cell Formation & Prevents Anemia:**
    *   Folic acid works with vitamin B12 to produce healthy red blood cells.
    *   A deficiency can lead to **megaloblastic anemia**, where red blood cells are abnormally large and immature, leading to sympt